# --- MKL-SVM Model Training ---

In [1]:
# --- General Imports ---
import numpy as np
import pandas as pd
import sys
from pathlib import Path
# --- Resolving Import Issue ---
notebook_dir = Path.cwd()
src_path = (notebook_dir / ".." / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --- Importing API Files ---
from mklsvm.svm import SVM

## > Info

In [2]:
help(SVM.fit)

Help on function fit in module mklsvm.svm.svm:

fit(
    self,
    X,
    Y,
    max_bo_iter=500,
    bo_tolerance=0.001,
    smo_c=1,
    max_smo_iter=2000,
    smo_tolerance=1e-05,
    general_timer=True,
    detailed_info=False
)
    #> Parameters Documentation:
    1. X: training x_array
    2. Y: training y_array
    3. max_bo_iter: Maximum iteration limit for "Beta" Optimization in MKL
    4. bo_tolerance: Minimum required changes in "Beta" in MKL
    5. smo_c: "C" parameter for SMO
    6. max_smo_iter: Iteration limit for optimizing "Alpha Vector" in SMO
    7. smo_tolerance: Minimum required changes in "Alpha Vector" in SMO
    8. general_timer: If True, will print the elapsed time of SVM.fit() function
    9. detailed_info: If True, will print the elapsed time of all underlaying functions of SVM.fit() function with their train details

    #> Attention: Be sure to call "SVM.select_kernels()", "SVM.kernels_arguments()" or "SVM.set_beta_optimizer()" before calling "SVM.fit()" fo

In [3]:
help(SVM.select_kernels)

Help on function select_kernels in module mklsvm.svm.svm:

select_kernels(self, selected_kernels=None, centering=True, normalization=True)
    #> Attention: Do Not call or give input if you wish to proceed with default selected kernels.

    Available Kernels: 1.Linear, 2.Polynomial, 3.RBF, 4.Laplacian, 5.Rational Quadratic, 6.Sigmoid
    Simply parse an array with the length of available kernels and set "True" for your desirable kernels you wish to be included in MKL.
    You can Enable/Disable kernel matrices "centering" and "normalization" by optional flag (Enabled by default).

    Default: [True, True, True, True, True, False]



In [4]:
help(SVM.set_kernels_arguments)

Help on function set_kernels_arguments in module mklsvm.svm.svm:

set_kernels_arguments(self, X, default_values=False)
    #> Attention: Do Not call or set "default_values=True" if you wish to proceed with default selected kernels.

    Simply run the method and answer the questions if you wish to modify the kernle's default values.



In [5]:
help(SVM.set_beta_optimizer)

Help on function set_beta_optimizer in module mklsvm.svm.svm:

set_beta_optimizer(self, method='Entropic Descent')
    #> Attention: Do Not call this function if you wish to proceed with default ("Entropic Descent") beta optimization algorithm.

    Available Beta Optimization Methods:
    "SimpleMKL", "L2-MKL", "Entropic Descent", "Hybrid L1/L2", "KTA", "Frank-Wolfe", "FISTA"



## > Breast Cancer Dataset 

In [6]:
#> --- Loading Processed Data --- <#
pca_cancer_x_train = np.load(r"..\\data\\splits\\cancer\\pca_cancer_x_train.npy", allow_pickle=True)
pca_cancer_x_test = np.load(r"..\\data\\splits\\cancer\\pca_cancer_x_test.npy", allow_pickle=True)
cancer_y_train = np.load(r"..\\data\\splits\\cancer\\cancer_y_train.npy")

In [7]:
# --- Model Instance & BO Method Selection ---
cancer_SVM_model = SVM()
cancer_SVM_model.select_kernels(centering = False)
cancer_SVM_model.set_beta_optimizer("Entropic Descent")

In [8]:
# --- Model Training ---
cancer_SVM_model.fit(pca_cancer_x_train, cancer_y_train,
                     max_bo_iter = 20, 
                     bo_tolerance = 1e-3,
                     smo_c = 1, 
                     max_smo_iter = 10**5, 
                     smo_tolerance = 1e-5, 
                     detailed_info=True)

Active Kernels: ['Linear' 'Polynomial' 'RBF' 'Laplacian' 'Rational Quadratic']
MKL.fit() Init Beta Array: [0.2 0.2 0.2 0.2 0.2]

SMO.fit() Elapsed Time Is 0:00:01.274128 On 276 Iteration With 82 Support Vectors.
SMO.fit() New b: 0.018188856467328075
MKL.fit() New Beta Array: [0.35738038 0.44193721 0.09822323 0.05445685 0.04800233]

SMO.fit() Elapsed Time Is 0:00:00.089670 On 1 Iteration With 82 Support Vectors.
SMO.fit() New b: 0.018188856467328075
MKL.fit() New Beta Array: [0.35738038 0.44193721 0.09822323 0.05445685 0.04800233]

SMO.fit() Elapsed Time Is 0:00:01.426671 On 340 Iteration With 77 Support Vectors.
SMO.fit() New b: 0.0025348655887766293
MKL.fit() Elapsed Time Is 0:00:02.980965 On 2 Iteration.

Dataset's Length: 456
Model's Support Vector Count: 77
MKL.fit() Final Beta Array: [0.35738038 0.44193721 0.09822323 0.05445685 0.04800233]


In [9]:
# --- Saving ---
cancer_svm_prediction = cancer_SVM_model.predict(pca_cancer_x_test)
cancer_SVM_model.save_model("..\\artifacts\\svm\\cancer\\")
np.save(r"..\\artifacts\\predictions\\cancer_svm_prediction", cancer_svm_prediction)

In [10]:
# --- Cleanup ---
del cancer_y_train, pca_cancer_x_test, pca_cancer_x_train
del cancer_SVM_model, cancer_svm_prediction

## > Ionosphere Dataset

In [11]:
#> --- Loading Processed Data --- <#
pca_ionosphere_x_train = np.load(r"..\\data\\splits\\ionosphere\\pca_ionosphere_x_train.npy", allow_pickle=True)
pca_ionosphere_x_test = np.load(r"..\\data\\splits\\ionosphere\\pca_ionosphere_x_test.npy", allow_pickle=True)
ionosphere_y_train = np.load(r"..\\data\\splits\\ionosphere\\ionosphere_y_train.npy")

In [12]:
# --- Model Instance & BO Method Selection ---
ionosphere_SVM_model = SVM()
ionosphere_SVM_model.set_beta_optimizer("Hybrid L1/L2")

In [13]:
# --- Model Training ---
ionosphere_SVM_model.fit(pca_ionosphere_x_train, ionosphere_y_train,
                         max_bo_iter = 20, 
                         bo_tolerance = 1e-3,
                         smo_c = 1,
                         max_smo_iter = 10**5,
                         smo_tolerance = 1e-5,
                         detailed_info=True)

Active Kernels: ['Linear' 'Polynomial' 'RBF' 'Laplacian' 'Rational Quadratic']
MKL.fit() Init Beta Array: [0.2 0.2 0.2 0.2 0.2]

SMO.fit() Elapsed Time Is 0:00:06.976773 On 1485 Iteration With 106 Support Vectors.
SMO.fit() New b: 0.2716253638143282
MKL.fit() New Beta Array: [0. 0. 0. 1. 0.]

SMO.fit() Elapsed Time Is 0:00:00.074339 On 1 Iteration With 106 Support Vectors.
SMO.fit() New b: 0.2716253638143282
MKL.fit() New Beta Array: [0. 0. 0. 1. 0.]

SMO.fit() Elapsed Time Is 0:00:03.563193 On 534 Iteration With 109 Support Vectors.
SMO.fit() New b: 0.303158882090818
MKL.fit() Elapsed Time Is 0:00:10.714243 On 2 Iteration.

Dataset's Length: 280
Model's Support Vector Count: 109
MKL.fit() Final Beta Array: [0. 0. 0. 1. 0.]


In [14]:
# --- Saving ---
ionosphere_svm_prediction = ionosphere_SVM_model.predict(pca_ionosphere_x_test)
ionosphere_SVM_model.save_model("..\\artifacts\\svm\\ionosphere\\")
np.save(r"..\\artifacts\\predictions\\ionosphere_svm_prediction", ionosphere_svm_prediction)

In [15]:
# --- Cleanup ---
del ionosphere_y_train, pca_ionosphere_x_test, pca_ionosphere_x_train
del ionosphere_SVM_model, ionosphere_svm_prediction

## > Pima Indians Diabetes Dataset

In [16]:
#> --- Loading Processed Data --- <#
pca_diabetes_x_train = np.load(r"..\\data\\splits\\diabetes\\pca_diabetes_x_train.npy", allow_pickle=True)
pca_diabetes_x_test = np.load(r"..\\data\\splits\\diabetes\\pca_diabetes_x_test.npy", allow_pickle=True)
diabetes_y_train = np.load(r"..\\data\\splits\\diabetes\\diabetes_y_train.npy")

In [17]:
# --- Model Instance & BO Method Selection ---
diabetes_SVM_model = SVM()
diabetes_SVM_model.set_beta_optimizer("KTA")

In [18]:
# --- Model Training ---
diabetes_SVM_model.fit(pca_diabetes_x_train, diabetes_y_train,
                       max_bo_iter = 20,
                       bo_tolerance = 1e-3,
                       smo_c = 1.0,
                       max_smo_iter = 10**5,
                       smo_tolerance = 1e-5,
                       detailed_info=True)

Active Kernels: ['Linear' 'Polynomial' 'RBF' 'Laplacian' 'Rational Quadratic']
MKL.fit() Init Beta Array: [0.2 0.2 0.2 0.2 0.2]

SMO.fit() Elapsed Time Is 0:00:19.237115 On 4368 Iteration With 340 Support Vectors.
SMO.fit() New b: -0.3934954159210899
MKL.fit() New Beta Array: [0.19581857 0.27776551 0.17455594 0.16243701 0.18942296]

MKL.fit() Elapsed Time Is 0:00:19.551315 On 1 Iteration.

Dataset's Length: 615
Model's Support Vector Count: 340
MKL.fit() Final Beta Array: [0.19581857 0.27776551 0.17455594 0.16243701 0.18942296]


In [19]:
# --- Saving ---
diabetes_svm_prediction = diabetes_SVM_model.predict(pca_diabetes_x_test)
diabetes_SVM_model.save_model("..\\artifacts\\svm\\diabetes\\")
np.save(r"..\\artifacts\\predictions\\diabetes_svm_prediction", diabetes_svm_prediction)

In [20]:
# --- Cleanup ---
del diabetes_y_train, pca_diabetes_x_test, pca_diabetes_x_train
del diabetes_SVM_model, diabetes_svm_prediction

## > Banknote Authentication Dataset

In [21]:
#> --- Loading Processed Data --- <#
pca_banknote_x_train = np.load(r"..\\data\\splits\\banknote\\pca_banknote_x_train.npy", allow_pickle=True)
pca_banknote_x_test = np.load(r"..\\data\\splits\\banknote\\pca_banknote_x_test.npy", allow_pickle=True)
banknote_y_train = np.load(r"..\\data\\splits\\banknote\\banknote_y_train.npy")

In [22]:
# --- Model Instance & BO Method Selection ---
banknote_SVM_model = SVM()
banknote_SVM_model.set_beta_optimizer("FISTA")

In [23]:
# --- Model Training ---
banknote_SVM_model.fit(pca_banknote_x_train, banknote_y_train,
                       max_bo_iter = 20,
                       bo_tolerance = 1e-3,
                       smo_c = 1,
                       max_smo_iter = 10**5,
                       smo_tolerance = 1e-5,
                       detailed_info=True)

Active Kernels: ['Linear' 'Polynomial' 'RBF' 'Laplacian' 'Rational Quadratic']
MKL.fit() Init Beta Array: [0.2 0.2 0.2 0.2 0.2]

SMO.fit() Elapsed Time Is 0:00:06.606711 On 2396 Iteration With 239 Support Vectors.
SMO.fit() New b: -0.21140891759026395
MKL.fit() New Beta Array: [0. 0. 0. 1. 0.]

SMO.fit() Elapsed Time Is 0:00:00.432186 On 1 Iteration With 239 Support Vectors.
SMO.fit() New b: -0.21140891759026395
MKL.fit() New Beta Array: [0. 0. 0. 1. 0.]

SMO.fit() Elapsed Time Is 0:01:29.743130 On 6694 Iteration With 258 Support Vectors.
SMO.fit() New b: -0.20085653354720298
MKL.fit() Elapsed Time Is 0:01:37.347638 On 2 Iteration.

Dataset's Length: 1098
Model's Support Vector Count: 258
MKL.fit() Final Beta Array: [0. 0. 0. 1. 0.]


In [24]:
# --- Saving ---
banknote_svm_prediction = banknote_SVM_model.predict(pca_banknote_x_test)
banknote_SVM_model.save_model("..\\artifacts\\svm\\banknote\\")
np.save(r"..\\artifacts\\predictions\\banknote_svm_prediction", banknote_svm_prediction)

In [25]:
# --- Cleanup ---
del banknote_y_train, pca_banknote_x_test, pca_banknote_x_train
del banknote_SVM_model, banknote_svm_prediction